In [11]:
import pandas as pd

df = pd.read_csv('data/thai_wikipron_5-4-2026.csv')
df

,writing,phonetic,count
0,ก,kɔː˧,2
1,ก,kɔː˧.kaj˨˩,2
2,ก.,kɔː˧,1
3,ก.ค.,kɔː˧.kʰɔː˧,1
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1
...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2
18307,ไฮ้,haj˦˥,1
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1


In [12]:
import re

def normalize_phonetic(phonetic: str) -> str:
    phonetic = (
        phonetic
        .replace('p̚', 'p')
        .replace('t̚', 't')
        .replace('k̚', 'k')
        .replace('a̯', 'ə')
        .replace('˥˩', '˦˩')
        .replace('˩˩˦', '˨˥')
    )

    SHORT_VOWELS = ['a', 'i', 'ɯ', 'u', 'e', 'ɤ', 'o', 'ɛ', 'ɔ']

    pattern = (
        '(' + '|'.join(map(re.escape, SHORT_VOWELS)) + ')'
        r'([˥˦˧˨˩]+)(?=\.|$)'
    )

    phonetic = re.sub(pattern, r'\1ʔ\2', phonetic)

    phonetic = re.sub(r'\.+', '.', phonetic)

    phonetic = re.sub(r'\.$', '', phonetic)

    return phonetic

In [13]:
df['normalized_phonetic'] = df['phonetic'].apply(normalize_phonetic)
df

,writing,phonetic,count,normalized_phonetic
0,ก,kɔː˧,2,kɔː˧
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩
2,ก.,kɔː˧,1,kɔː˧
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧
...,...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2,haj˧.droː˧.t͡ɕeːn˧
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩
18307,ไฮ้,haj˦˥,1,haj˦˥
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,….daj˧.….nɯŋ˨˩


In [14]:
import thai_gpa

def evaluate(text: str, ipa: str) -> tuple:
    result = thai_gpa.align(text, ipa)
    reconstructed_text = ''.join(s.reconstruct_text() for s in result)
    reconstructed_ipa = '.'.join(s.get_ipa(is_reduplicated=s.is_reduplicated) for s in result)
    return reconstructed_text, reconstructed_ipa

print(evaluate('การขัดกันของผลประโยชน์', 'kaːn˧.kʰat˨˩.kan˧.kʰɔːŋ˨˥.pʰon˨˥.praʔ˨˩.joːt˨˩'))

('การขัดกันของผลประโยชน์', 'kaːn˧.kʰat˨˩.kan˧.kʰɔːŋ˨˥.pʰon˨˥.praʔ˨˩.joːt˨˩')


In [15]:
from tqdm.auto import tqdm
tqdm.pandas()

def apply_evaluate(row):
    try:
        reconstructed_text, phonetic_answer = evaluate(row['writing'], row['normalized_phonetic'])
    except Exception as e:
        reconstructed_text, phonetic_answer = None, f'ERROR: {e}'
    return pd.Series([reconstructed_text, phonetic_answer])

df[['reconstructed_text', 'phonetic_answer']] = df.progress_apply(apply_evaluate, axis=1)
df.to_csv('data/test.csv', index=False)
df

 91%|█████████ | 16680/18310 [04:36<00:17, 91.66it/s] c:\Storage\repos\thai_grapheme_sandbox\thai_ipa.py:109: UserWarning: Warning: No explicit glottal stop at "e"
  warnings.warn(f'Warning: No explicit glottal stop at "{original}"')
100%|██████████| 18310/18310 [04:57<00:00, 61.52it/s] 


,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,None,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,None,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,None,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,None,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,None,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2,haj˧.droː˧.t͡ɕeːn˧,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˦˩
18307,ไฮ้,haj˦˥,1,haj˦˥,ไฮ้,haj˦˥
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,….daj˧.….nɯŋ˨˩,None,ERROR: argument of type 'NoneType' is not iter...


In [16]:
mask = ~df["phonetic_answer"].str.startswith("ERROR:")
mismatches = df.loc[
    mask & (df["normalized_phonetic"] != df["phonetic_answer"]),
    ["writing", "normalized_phonetic", "phonetic_answer"]
]

mismatches

,writing,normalized_phonetic,phonetic_answer


In [17]:
errors = df[df["phonetic_answer"].str.startswith("ERROR:")]
errors

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,None,ERROR: Could not align 'ก' with 'kɔː˧'
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,None,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,None,ERROR: Could not align 'ก.' with 'kɔː˧'
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,None,ERROR: Could not align 'ก.ค.' with 'kɔː˧.kʰɔː˧'
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,None,ERROR: Could not align 'ก.ท.ม.' with 'kɔː˧.tʰɔ...
...,...,...,...,...,...,...
18263,ไอดอล,ʔaj˧.dɔl˥˩,2,ʔaj˧.dɔl˦˩,None,ERROR: argument of type 'NoneType' is not iter...
18278,ไอศครีม,ʔajs˧.kʰriːm˧,2,ʔajs˧.kʰriːm˧,None,ERROR: argument of type 'NoneType' is not iter...
18279,ไอศวรรย์,ʔaj˧.sa˨˩.wan˩˩˦,1,ʔaj˧.saʔ˨˩.wan˨˥,None,ERROR: Could not align 'ไอศวรรย์' with 'ʔaj˧.s...
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,….daj˧.….nɯŋ˨˩,None,ERROR: argument of type 'NoneType' is not iter...


In [18]:
errors.sample(10)   

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
13310,อย่า,jaː˨˩,1,jaː˨˩,None,ERROR: Could not align 'อย่า' with 'jaː˨˩'
8134,พล,pʰɔː˧.lɔː˧,4,pʰɔː˧.lɔː˧,None,ERROR: Could not align 'พล' with 'pʰɔː˧.lɔː˧'
12385,ห,hɔː˩˩˦,2,hɔː˨˥,None,ERROR: Could not align 'ห' with 'hɔː˨˥'
5818,นฤบดินทร,na˦˥.rɯ˦˥.bɔː˧.din˧,1,naʔ˦˥.rɯʔ˦˥.bɔː˧.din˧,None,ERROR: Could not align 'นฤบดินทร' with 'naʔ˦˥....
1611,กุญแจซอล,kun˧.t͡ɕɛː˧.sɔl˧,2,kun˧.t͡ɕɛː˧.sɔl˧,None,ERROR: argument of type 'NoneType' is not iter...
322,กระเชอก้นรั่ว,kra˨˩.t͡ɕʰɤː˧.kon˥˩.rua̯˥˩,1,kraʔ˨˩.t͡ɕʰɤː˧.kon˦˩.ruə˦˩,None,ERROR: Could not align 'กระเชอก้นรั่ว' with 'k...
17094,แอสทาทีน,ʔɛːs˦˥.tʰaː˧.tʰiːn˧,1,ʔɛːs˦˥.tʰaː˧.tʰiːn˧,None,ERROR: argument of type 'NoneType' is not iter...
14802,เซลล์ไข่,seːl˧.kʰaj˨˩,1,seːl˧.kʰaj˨˩,None,ERROR: argument of type 'NoneType' is not iter...
14099,ฮันกึล,han˧.kɯl˧,1,han˧.kɯl˧,None,ERROR: argument of type 'NoneType' is not iter...
14778,เช่าทรัพย์สิน,t͡ɕʰaw˥˩.sap̚˦˥.sin˩˩˦,1,t͡ɕʰaw˦˩.sap˦˥.sin˨˥,None,ERROR: Could not align 'เช่าทรัพย์สิน' with 't...
